In [1]:
from Objects.Transformations import *
from Objects.WSBM import *
from Objects.TWSBMInstance import *

from Computation.Computation import *
from Computation.ExtraMetrics import *

from Plotting.Plotting import *
from Plotting.ArtisticPlotting import *

In [2]:
def plot_embedding_for(transforms, subfolder):
	for emb_mode, p22 in product(EMB_MODES[:1], P22S):
		print(f"Simulating for emb_mode = {emb_mode}, p22 = {p22}")
		metrics = {}
		for rho, pi in product(RHOS, PIS):
			metrics[(rho, pi)] = {}
			for model, model_params in MODELS_AND_PARAMS:
				m = model(rho, pi, model_params, p22 = p22)
				A, Z = m(42)
				metrics[(rho, pi)][m] = {}
				for t in transforms:
					#print(f"Simulating for rho={rho}, pi={pi}, model={model.name}, transformation={t.name}")
					metrics[(rho, pi)][m][t] = TWSBMInstance(model = m, transformation = t, A = t(A), Z = Z, emb_mode = emb_mode)

		plotter = Plotter(folder_path=emb_mode_p22_path_str(emb_mode, p22))
		for rho, pi in product(RHOS, PIS):
			plotter.plot_embedding(rho, pi, metrics[(rho, pi)], subfolder = subfolder)

plot_embedding_for(TRANSFORMS, "BetaLog/Transforms")
plot_embedding_for(TRANSFORMS_POW, "BetaLog/Powers")
plot_embedding_for(TRANSFORMS_QTL, "BetaLog/Quantiles")

Simulating for emb_mode = sqrt-scaled, p22 = fixed
Simulating for emb_mode = sqrt-scaled, p22 = p11
Simulating for emb_mode = sqrt-scaled, p22 = fixed
Simulating for emb_mode = sqrt-scaled, p22 = p11
Simulating for emb_mode = sqrt-scaled, p22 = fixed
Simulating for emb_mode = sqrt-scaled, p22 = p11


In [3]:
def plot_embedding_for(transforms, subfolder):  
	for emb_mode, p22 in product(EMB_MODES[:1], P22S):
		print(f"Simulating for emb_mode = {emb_mode}, p22 = {p22}")
		metrics = {}
		for rho, pi in product(RHOS, PIS):
			metrics[(rho, pi)] = {}
			for model, model_params in product([lognormWSBM], [(1, 1), (0.5, 1), (1, 0.5), (0.1, 0.5)]):
			#for model, model_params in product([lognormWSBM], [(0.15, 0.17), (0.15, 0.19), (0.15, 0.21), (0.15, 0.23)]):
				m = model(rho, pi, model_params, p22 = p22)
				A, Z = m(42)
				metrics[(rho, pi)][m] = {}
				for t in transforms:
					#print(f"Simulating for rho={rho}, pi={pi}, model={model.name}, transformation={t.name}")
					metrics[(rho, pi)][m][t] = TWSBMInstance(model = m, transformation = t, A = t(A), Z = Z, emb_mode = emb_mode)

		plotter = Plotter(folder_path=emb_mode_p22_path_str(emb_mode, p22))
		for rho, pi in product(RHOS, PIS):
			plotter.plot_embedding(rho, pi, metrics[(rho, pi)], subfolder = subfolder)
plot_embedding_for(TRANSFORMS, "Log/Transforms")
plot_embedding_for(TRANSFORMS_POW, "Log/Powers")
plot_embedding_for(TRANSFORMS_QTL, "Log/Quantiles")

Simulating for emb_mode = sqrt-scaled, p22 = fixed
Simulating for emb_mode = sqrt-scaled, p22 = p11
Simulating for emb_mode = sqrt-scaled, p22 = fixed
Simulating for emb_mode = sqrt-scaled, p22 = p11
Simulating for emb_mode = sqrt-scaled, p22 = fixed
Simulating for emb_mode = sqrt-scaled, p22 = p11


In [2]:
emb_mode = 'sqrt-scaled'
p22 = 'fixed'
n_batch = 6

In [3]:
path = f"Computation/{emb_mode_p22_path_str(emb_mode, p22)}_old"
metrics_g = {}
metrics_g_1st_layer = {}
for rho, pi, model in RHOS_PIS_MODELS:
	file = f"{path}/{model.__name__}_{rho}_{pi}".replace(".", "")
	grids_stacked = [np.load(f"{file}/{b}.npz") for b in range(n_batch)]
	metrics_g[(rho, pi, model)] = {}
	metrics_g_1st_layer[(rho, pi, model)] = {}
	for t in TRANSFORMS:
		metrics_g[(rho, pi, model)][t] = {}
		metrics_g[(rho, pi, model)][t]['std'] = {}
		metrics_g_1st_layer[(rho, pi, model)][t] = {}
		for metric in METRICS_ID:
			g_stack = np.concatenate([g[f'{t.id}_{metric}'] for g in grids_stacked], axis = -1)
			g_stack[g_stack > 1e300] = 0
			mean = np.mean(g_stack, axis = -1)
			std  = np.std(g_stack, axis = -1)
			metrics_g[(rho, pi, model)][t][metric] = mean
			metrics_g[(rho, pi, model)][t]['std'][metric] = std
			metrics_g_1st_layer[(rho, pi, model)][t][metric] = g_stack[:, :, 0]

metrics_g = aggregate_metrics(metrics_g)
metrics_g_1st_layer = aggregate_metrics(metrics_g_1st_layer)

metrics_g = best_transform_metrics(metrics_g)

for model in MODELS:
	metrics_g[model] = best_transform_metrics(metrics_g[model])

for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	metrics_g[(rho, pi, model)] = best_transform_metrics(m)
	for t in TRANSFORMS:
		m = metrics_g[(rho, pi, model)][t]
		metrics_g[(rho, pi, model)][t] = correlation(m)
		metrics_g[(rho, pi, model)][t] = bias(m)

plotter = Plotter(folder_path=emb_mode_p22_path_str(emb_mode, p22))

c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Nicol\AppData\Local\Pro

In [4]:
plotter.plot_scatter_Rand_vs_Chernoff(metrics_g_1st_layer, n_points_ratio_displayed=0.25)

In [7]:
for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	for t in TRANSFORMS:
		m = metrics_g[(rho, pi, model)][t]
		plotter.plot_metrics_heatmap(rho, pi, model, t, m, shared=False, log=True)

In [ ]:
for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	for t in TRANSFORMS:
		m = metrics_g[(rho, pi, model)][t]
		plotter.plot_bias_heatmap(rho, pi, model, t, m, log = True)

In [5]:
for rho, pi, model in RHOS_PIS_MODELS:
	m = metrics_g[(rho, pi, model)]
	plotter.plot_best_transform_heatmaps(rho, pi, model, m)

In [ ]:
# TODO : std (en vérité moyenne des stds)

for model in MODELS:
	plotter.plot_transforms_rand(model, metrics_g[model], mode = 'No regret')
	plotter.plot_transforms_rand(model, metrics_g[model], mode = 'With regret')

	for chernoff in CHERNOFFS_ID:
		plotter.plot_transforms_rand_for_best_transform(model, metrics_g[model], chernoff)

In [4]:
metrics_l = {'p11' : {}, 'p12' : {}}
N = metrics_g[RHOS_PIS_MODELS[0]][TRANSFORMS[0]][METRICS_ID[0]].shape[0]

for p in ['p11', 'p12']:
	for i in range(N):
		metrics_l[p][i] = {}
		for rho, pi, model in RHOS_PIS_MODELS:
			mg_rpm  = metrics_g[(rho, pi, model)]
			metrics_l[p][i][(rho, pi, model)] = {}
			for t in TRANSFORMS:
				mg_rpm_t = mg_rpm[t]
				metrics_l[p][i][(rho, pi, model)][t] = {}
				metrics_l[p][i][(rho, pi, model)][t]['std'] = {}
				for m in METRICS_ID:
					if p == 'p11':
						mean = mg_rpm_t[m][:, i]
						std  = mg_rpm_t['std'][m][:, i]
					else:
						mean = mg_rpm_t[m][i, :]
						std  = mg_rpm_t['std'][m][i, :]
					metrics_l[p][i][(rho, pi, model)][t][m]        = mean
					metrics_l[p][i][(rho, pi, model)][t]['std'][m] = std

			m = metrics_l[p][i][(rho, pi, model)]
			metrics_l[p][i][(rho, pi, model)] = best_transform_metrics(m)

c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Nicol\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [5]:
for rho, pi, model in RHOS_PIS_MODELS:
	plotter.plot_line_sliding(plotter.plot_best_transform_lines, 
								   rho, pi, model, metrics_l, p22 = 'fixed', n = n, slider = 'p11')
	plotter.plot_line_sliding(plotter.plot_best_transform_lines, 
								   rho, pi, model, metrics_l, p22 = 'fixed', n = n, slider = 'p12')

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='α₁₁', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='α₁₂', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='σ₁₁', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='σ₁₂', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='α₁₁', max=0.9…

...

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='σ₁₂', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='α₁₁', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='α₁₂', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='σ₁₁', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='σ₁₂', max=0.9…

In [5]:
for rho, pi, model in RHOS_PIS_MODELS:
	plotter.plot_line_sliding(plotter.plot_chernoffs_lines, 
								   rho, pi, model, metrics_l, p22 = 'fixed', n = n, slider = 'p11')
	plotter.plot_line_sliding(plotter.plot_chernoffs_lines, 
								   rho, pi, model, metrics_l, p22 = 'fixed', n = n, slider = 'p12')

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='α₁₁', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='α₁₂', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='σ₁₁', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='σ₁₂', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='α₁₁', max=0.9…

...

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='σ₁₂', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='α₁₁', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='α₁₂', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='σ₁₁', max=0.9…

interactive(children=(FloatSlider(value=0.504950495049505, continuous_update=False, description='σ₁₂', max=0.9…